# Embeddings

임베딩은 텍스트의 의미를 실수 벡터로 바꾸는 표현 방식이다. 의미가 가까운 문장은 벡터 공간에서도 가까워지므로 검색과 추천에 사용할 수 있다.

이 노트북에서는 문장 벡터를 만든다. 리뷰 벡터를 저장한 뒤 질의 벡터와의 cosine similarity로 가까운 리뷰를 찾는다.


## 모델과 벡터 차원

이 노트북은 비용과 저장 공간을 고려해 `text-embedding-3-small`을 사용한다.

- `text-embedding-3-small`의 기본 출력은 1,536차원이다.
- `text-embedding-3-large`의 기본 출력은 3,072차원이다.
- 두 모델 모두 `dimensions`로 출력 차원을 줄일 수 있다.
- 차원을 줄이면 저장·검색 비용과 검색 품질이 함께 달라질 수 있다.
- 최종 모델과 차원은 실제 서비스 검증 데이터로 결정한다.

공식 문서는 다음과 같다.

- [Embeddings 가이드](https://developers.openai.com/api/docs/guides/embeddings)
- [Embeddings 생성 API](https://developers.openai.com/api/reference/resources/embeddings/methods/create)
- [text-embedding-3-small 모델 문서](https://developers.openai.com/api/docs/models/text-embedding-3-small)


## MTEB 벤치마크 참고

MTEB는 분류, 군집화, 검색, 의미 유사도 등 여러 임베딩 작업을 평가하는 벤치마크 모음이다. 리더보드는 데이터셋과 평가 구성에 따라 달라지므로 특정 모델의 고정 순위를 보장하지 않는다.

- 검색 과제는 nDCG@k, 재정렬 과제는 MRR@k 또는 MAP 같은 지표를 사용한다.
- 서비스에서 사용할 언어, 문서 길이, 질의 유형과 유사한 검증 데이터로 모델을 비교한다.
- MTEB 리더보드: https://huggingface.co/spaces/mteb/leaderboard


### API 클라이언트 준비

이미 설정한 `.env`를 불러온다. 키 값은 출력하지 않는다.


In [1]:
import os

from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("08_llm 프로젝트 최상위의 .env 파일을 확인한다.")
load_dotenv(dotenv_path, override=False)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env의 OPENAI_API_KEY를 확인한다.")


### 단일 문장을 1,536차원 벡터로 변환하기

문자열 하나를 길이 1의 목록으로 `input`에 전달하면 응답의 `data[0].embedding`에 실수 벡터가 들어간다. 이 벡터는 다음 셀의 배치 변환 함수와 같은 형식이다.

모델 문서: https://developers.openai.com/api/docs/models/text-embedding-3-small


In [4]:
from openai import OpenAI

# OpenAI()는 환경 변수에서 인증 정보를 읽는다.
client = OpenAI()
text = '임베딩 예시 문장'

response = client.embeddings.create(
    model='text-embedding-3-small',
    input=[text],
)
# data[0]은 입력 목록의 첫 문장에 해당한다.
embedding = response.data[0].embedding
print(len(embedding))
print(embedding[0:10])
# 0.050323486328125, 0.035247802734375, 0.0222930908203125

1536
[0.050323486328125, 0.035247802734375, 0.0222930908203125, -0.0066986083984375, -0.0245208740234375, 0.0153656005859375, -0.0157318115234375, -0.038482666015625, -0.0202789306640625, -0.031585693359375]


### 여러 문장을 한 번에 임베딩하기

문자열 목록을 줄바꿈만 공백으로 바꿔 의미를 보존한 채 API에 전달한다. 응답 벡터의 순서는 입력 문장 순서와 같으며, 결과는 리뷰 데이터의 `embedding` 열에 저장된다.

특수문자나 개행을 일괄 삭제하면 품질이 높아진다고 단정할 수 없다. 서비스 데이터의 의미를 유지하는 정규화만 적용하고 검색 결과로 효과를 확인한다.


In [6]:
import numpy as np

def texts_to_embedding(texts, model='text-embedding-3-small'):
    # 줄바꿈만 공백으로 변환하여 문장 경계를 유지한 입력 목록 만들기
    normalized_texts = [text.replace('\n', ' ') for text in texts]

    # 각 입력 문장과 같은 순서의 embedding 목록을 응답으로 받기
    response = client.embeddings.create(
        model=model,
        input=normalized_texts,
    )

    # response.data: 문장 묶음 -> 벡터 묶음
    return [item.embedding for item in response.data]


# 2문장을 입력하여 결과 반환받기
sample_texts = [
    "hello world",
    "goodbye world",
]

output = texts_to_embedding(sample_texts)
print(np.array(output).shape) # (문장:2, 벡터:1536)

(2, 1536)


## 음식 리뷰 유사도 검색

검색은 문서와 질의를 같은 모델로 임베딩한 뒤 cosine similarity가 높은 문서를 반환하는 과정이다. 아래에서는 리뷰의 제목과 본문을 결합해 문서 하나당 벡터 하나를 만든다.


### 리뷰 CSV 내려받기

이 셀은 외부 파일 ID에서 `fine_food_reviews_1k.csv`를 받는다. 네트워크와 외부 파일 상태에 따라 다운로드가 실패할 수 있다.

다운로드가 끝나면 다음 셀의 `read_csv`가 같은 이름의 CSV를 DataFrame으로 읽는다.


In [12]:
%pip install gdown

  Using cached gdown-6.1.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
Using cached gdown-6.1.0-py3-none-any.whl (19 kB)
Using cached PySocks-1.7.1-py3-none-any.whl (16 kB)

   ------------- -------------------------- 1/3 [filelock]
   ------------- -------------------------- 1/3 [filelock]
   ------------- -------------------------- 1/3 [filelock]
   ------------- -------------------------- 1/3 [filelock]
   ------------- -------------------------- 1/3 [filelock]
   -------------------------- ------------- 2/3 [gdown]
   -------------------------- ------------- 2/3 [gdown]
   -------------------------- ------------- 2/3 [gdown]
   ---------------------------------------- 3/3 [gdown]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import subprocess

subprocess.run(
    [
        "gdown",
        "--output",
        "fine_food_reviews_1k.csv",
        "1tSQZQFYD64_mrL9CjDcn6KruZp7_smuD",
    ],
    check=True,
)


CompletedProcess(args=['gdown', '--output', 'fine_food_reviews_1k.csv', '1tSQZQFYD64_mrL9CjDcn6KruZp7_smuD'], returncode=0)

### 리뷰 행을 DataFrame으로 읽기

CSV의 각 행은 리뷰 하나이며 `Summary`와 `Text` 열이 다음 결합 단계의 입력이다. `head()`는 열 이름과 일부 행을 보여 주어 문자열 열이 올바르게 읽혔는지 확인한다.


In [15]:
import pandas as pd

# 첫 번째 CSV 열은 행 인덱스로 사용한다.
df = pd.read_csv('fine_food_reviews_1k.csv', index_col=0)
df.head()


,Time,ProductId,UserId,Score,Summary,Text,n_tokens
Unnamed: 0,,,,,,,
0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,33
1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",26
2,1351123200,B000JMBE7M,AQX1N6A51QOKG,4,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...,242
3,1351123200,B004AHGBX4,A2UY46X0OSNVUQ,3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...,216
4,1351123200,B001BORBHO,A1AFOYZ9HSM2CZ,5,Happy with the product,My dog was suffering with itchy skin. He had ...,76


### 리뷰 데이터의 열과 결측치 확인

`info()`는 행 수, 열 이름, 결측이 아닌 값의 수를 보여 준다. `Summary` 또는 `Text`에 결측치가 있으면 문자열 결합 전에 처리 규칙을 정해야 한다.


In [16]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Time       1000 non-null   int64
 1   ProductId  1000 non-null   str  
 2   UserId     1000 non-null   str  
 3   Score      1000 non-null   int64
 4   Summary    1000 non-null   str  
 5   Text       1000 non-null   str  
 6   n_tokens   1000 non-null   int64
dtypes: int64(3), str(4)
memory usage: 54.8 KB


### 제목과 본문을 하나의 검색 문서로 결합하기

`str.strip()`은 앞뒤 공백만 정리하고 제목과 본문 사이에는 `; `를 넣는다. 생성한 `combined` 문자열은 다음 API 요청의 입력이며, 나중에는 검색 결과의 인덱스가 된다.


In [17]:
# 제목과 본문을 구분자로 이어 리뷰당 검색 문서 하나를 만든다.
df['combined'] = df['Summary'].str.strip() + '; ' + df['Text'].str.strip()
df[['combined']].head()


,combined
Unnamed: 0,
0,where does one start...and stop... with a tre...
1,Arrived in pieces; Not pleased at all. When I ...
2,"It isn't blanc mange, but isn't bad . . .; I'm..."
3,These also have SALT and it's not sea salt.; I...
4,Happy with the product; My dog was suffering w...


### 리뷰 문서를 벡터 열로 저장하기

`combined` 열의 문자열 목록을 배치로 보내면 각 행에 대응하는 임베딩 목록이 돌아온다. 이 API 호출은 네트워크와 인증이 필요하며, 반환한 벡터 열은 다음 검색 인덱스의 입력이다.


In [19]:

df['embedding'] = texts_to_embedding(df['combined'].tolist())
df[['combined', 'embedding']].head()

,combined,embedding
Unnamed: 0,,
0,where does one start...and stop... with a tre...,"[0.0301055908203125, -0.02056884765625, -0.028..."
1,Arrived in pieces; Not pleased at all. When I ...,"[0.01125335693359375, 0.034912109375, -0.03875..."
2,"It isn't blanc mange, but isn't bad . . .; I'm...","[0.0022869110107421875, 0.00484466552734375, -..."
3,These also have SALT and it's not sea salt.; I...,"[-0.0157928466796875, 0.013763427734375, -0.02..."
4,Happy with the product; My dog was suffering w...,"[0.00012731552124023438, -0.0675048828125, 0.0..."


### 저장된 벡터의 값과 길이 살펴보기

이 셀은 첫 리뷰 벡터를 확인하는 용도이다. 전체 벡터 목록을 출력하면 너무 길어지므로 첫 행의 길이와 앞부분만 확인한다.


In [20]:
first_embedding = df['embedding'].iloc[0]
print(len(first_embedding), first_embedding[:5])


1536 [0.0301055908203125, -0.02056884765625, -0.0286102294921875, 0.022125244140625, -0.028717041015625]


### 검색용 벡터 인덱스 만들기

`embed_df`는 벡터를 열로, 결합 리뷰 문장을 인덱스로 둔다. 다음 검색 함수는 이 DataFrame에 문서별 cosine similarity 열을 추가하고 상위 행을 반환한다.


In [23]:
# 리뷰 원문을 인덱스로 두어 검색 결과에서 바로 읽는다.
embed_df = df[['embedding']].copy()
embed_df.index = df['combined']
display(embed_df.head())
print(embed_df.shape)


,embedding
combined,
where does one start...and stop... with a treat like this; Wanted to save some to bring to my Chicago family but my North Carolina family ate all 4 boxes before I could pack. These are excellent...could serve to anyone,"[0.0301055908203125, -0.02056884765625, -0.028..."
"Arrived in pieces; Not pleased at all. When I opened the box, most of the rings were broken in pieces. A total waste of money.","[0.01125335693359375, 0.034912109375, -0.03875..."
"It isn't blanc mange, but isn't bad . . .; I'm not sure that custard is really custard without eggs. But this comes close. I got it for use in a ""Vegan pancake"" recipe. We were having houseguests who were Vegan and I wanted to make some special breakfasts while they were here. One of the cooking/recipe sites had a recipe using this and there were lots of great reviews. I tried the recipe and it turned out like wallpaper paste -- yuck!<br />However, the so-called custard isn't so bad. I think it's probably just cornstarch and annatto (yellow coloring with a slight flavor). It's fun playing with it. You could dress it up with fruit. Seems to come out on the thin side when you make it as directed, so I use less milk because I like my custards to set firm. As a custard sauce it's fine. I would say it tastes something between a pudding and a custard.<br /><br />If you want a really good egg-free ""custard"" get an original recipe for ""blanc mange."" It takes a lot longer to make, but it's certainly worth the difference.","[0.0022869110107421875, 0.00484466552734375, -..."
"These also have SALT and it's not sea salt.; I like the fact that you can see what you're getting and that there are no bones or dark meat. There are 7 nice big chunks in every jar.<br /><br />These taste like tuna in a can but, because they're preserved in glass, you don't have to worry about either aluminum or BPA; BUT ... they are not just tuna and spring water.<br /><br />There is salt in there, too, and it's not healthy sea salt, it's toxic table salt.<br /><br />I am trying to contact Tonnino to confirm that. I might be wrong because the label states that the ingredients are ""tuna fish"" but the sticker on the top clarifies that it is the smaller (healthier) yellowfin, so the ""salt"" listed in the ingredients might be sea salt but, if it was, why don't they say so?<br /><br />Without confirmation, I will continue to look for a salt-free olive-oil free tuna preserved in glass.<br /><br />If you know of one, please contact me!","[-0.0157928466796875, 0.013763427734375, -0.02..."
Happy with the product; My dog was suffering with itchy skin. He had been eating Natural Choice brand (cheaper) since he was a puppy. I was nervous to change foods. The vet suggested to change foods sand see if the skin issues cleared up. Wellness brand did the job. My dog seems to love the food and the skin issues cleared up within a few weeks.,"[0.00012731552124023438, -0.0675048828125, 0.0..."


(1000, 1)


### cosine similarity로 상위 리뷰 찾기

cosine similarity는 두 벡터 방향의 유사도를 계산하며 1에 가까울수록 방향이 비슷하다. 질의와 문서는 반드시 같은 임베딩 모델로 변환해야 차원과 벡터 공간이 일치한다.

함수는 질의 문자열을 1개 벡터로 바꾸고, 모든 문서 벡터와의 점수를 계산한 뒤 높은 순서로 `top_n`개를 반환한다.


In [28]:
from sklearn.metrics.pairwise import cosine_similarity

# embed_df에서 query와 코사인 유사도가 가장 높은 5개 조회
def review_search(query, embed_df, top_n=5):

    # 평문 query -> 임베딩 벡터화 query
    query_embedding = texts_to_embedding([query])

    # 원본 유지를 위해서 복사본 생성
    scored = embed_df.copy()

    # 각 문서 벡터(리뷰)와 query 벡터를 입력해서 코사인 유사도를 구해
    # scored 데이터프레임의 'cos_sim' 컬럼에 추가
    scored['cos_sim'] = scored['embedding'].apply(
        # query_embedding -> 2차원
        # scored['embedding'] 컬럼값 1개 == 문장 벡터 == x == 1차원
        # -> 차원 수를 맞추기 위해서 [x] 형태로 변경
        lambda x: cosine_similarity(query_embedding, [x])[0,0]
    )

    # 코사인 유사도 점수가 높은 순서로 정렬 후 상위 top_n 만큼 반환
    return scored.sort_values('cos_sim', ascending=False).head(top_n)


# 테스트
review_search('delicious fruit', embed_df)

,embedding,cos_sim
combined,,
"Delicious .; These plums are sweet and juicy, and the aroma is like perfume. And it doesn't hurt that they are good for you, too.","[0.03155517578125, -0.0019989013671875, -0.058...",0.615371
"Delicious!; For anyone who says ""I don't like fruitcake"" or anyone who's never had fruitcake and wonders what all the fuss is about, try this. (As long as you're not allergic to tree nuts or any other ingredient.) It's chock-a-block with nuts and moist fruit. I will definitely be buying more.","[-0.0166473388671875, 0.002017974853515625, -0...",0.602412
"These are Delicious!; Great taste, right price, fabulous snack! It is the only fruit I can get my little one to eat and I can't keep my high school son out of them either. They are great pick-me-ups on the way to my daughter's soccer practice or before my early morning run. And best of all, Amazon ships these right to my door every month. No more finding the right store who carries them. I just set up the automatic recurring shipment once and it works like a charm!","[0.028717041015625, -0.01357269287109375, -0.0...",0.583662
"Delicious!; Wonderful! Deep, rich, pure black raspberry syrup! Absolutely delicious on waffles, cheesecake, ice cream, yogurt, drinks, etc. Thrilled to see that there are at least some berry syrup makers who do not feel the need to ""sour"" the flavor of perfect berry products with citric acid!","[0.00879669189453125, -0.037872314453125, -0.0...",0.536596
"What a treat!; Ordered these as part of a presentation on Malaysia as this fruit had left a very positive impression on me during my visit. We gave these to 4th graders who were equally impressed with the exterior soft spikes and the juicy center. The fruits arrived in good condition, just slightly bruised. The kept well in the fridge in a single-double layer covered with a damp paper towel. I had also rinsed in a vinegar-water (1:10 parts) solution to be sure to inhibit any mold (like I do with berries). They kept from Saturday when they arrived to Thursday with minimal browning. I ordered 4 pounds and had plenty of fruits for the 40 kids. I think there were about 55 - 60 fruits all together. It was a fantastic treat and the kids were requesting we bring them again for the Halloween party! I'm sure they'll remember the presentation for a long time.","[-0.02532958984375, -0.0016870498657226562, -0...",0.485506


### 다른 질의로 검색 결과 비교하기

같은 저장 벡터에 다른 질의를 넣으면 문서 임베딩을 다시 만들지 않고 순위만 다시 계산한다. 오탈자나 짧은 질의에서 결과가 약하면 질의 정규화, 데이터 품질, 모델 선택을 검증 데이터로 점검한다.


In [29]:
review_search('bad delivery', embed_df)


,embedding,cos_sim
combined,,
"great product, poor delivery; The coffee is excellent and I am a repeat buyer. Problem this time was with the UPS delivery. They left the box in front of my garage door in the middle of the driveway. Because of this odd delivery location, my wife ran over the box when she backed out of the garage and did not see the box. we lost half of the cups. Thius is the third time I have written about this matter to Amazon with no results. Hopefully someone will respond to me.<br /><br />fred santaniello","[-0.032958984375, -0.010345458984375, 0.007141...",0.511761
"great product, poor delivery; The coffee is excellent and I am a repeat buyer. Problem this time was with the UPS delivery. They left the box in front of my garage door in the middle of the driveway. Because of this odd delivery location, my wife ran over the box when she backed out of the garage and did not see the box. we lost half of the cups. Thius is the third time I have written about this matter to Amazon with no results. Hopefully someone will respond to me.<br /><br />fred santaniello","[-0.032958984375, -0.010345458984375, 0.007141...",0.511761
"great product, poor delivery; The coffee is excellent and I am a repeat buyer. Problem this time was with the UPS delivery. They left the box in front of my garage door in the middle of the driveway. Because of this odd delivery location, my wife ran over the box when she backed out of the garage and did not see the box. we lost half of the cups. Thius is the third time I have written about this matter to Amazon with no results. Hopefully someone will respond to me.<br /><br />fred santaniello","[-0.03289794921875, -0.0103302001953125, 0.007...",0.511728
"great product, poor delivery; The coffee is excellent and I am a repeat buyer. Problem this time was with the UPS delivery. They left the box in front of my garage door in the middle of the driveway. Because of this odd delivery location, my wife ran over the box when she backed out of the garage and did not see the box. we lost half of the cups. Thius is the third time I have written about this matter to Amazon with no results. Hopefully someone will respond to me.<br /><br />fred santaniello","[-0.032958984375, -0.01029205322265625, 0.0071...",0.511572
Disappointed; The metal cover has severely disformed. And most of the cookies inside have been crushed into small pieces. Shopping experience is awful. I'll never buy it online again.,"[0.007610321044921875, -0.00868988037109375, -...",0.377094
